# 과제 261106

## import 판다스, 시봉~

In [217]:
import pandas as pd                                  # 판다스 불러오기
import seaborn as sns                                # seaborn 불러오기 (titanic 데이터 다운로드용)

## 데이터 다운로드(한번만 실행)

In [218]:
# 실행 전 준비: VS Code 탐색기에서 마우스로 pandas_prj 폴더 안에 data → raw 폴더를 만들어 두세요.
# seaborn이 없다는 에러가 나면 터미널(.venv 활성화 상태)에서 pip install seaborn 실행

titanic_raw = sns.load_dataset("titanic")            # seaborn 예제 데이터 titanic 다운로드
titanic_raw.to_csv("data/raw/titanic.csv", index=False)  # data/raw 폴더에 csv로 저장
print("저장 완료")

저장 완료


## 데이터 불러오기

In [219]:
titanic = pd.read_csv("data/raw/titanic.csv")      # 저장한 파일 불러오기
print(titanic.shape)                                 # (891, 15)가 나오면 성공
print(titanic.head(3))

(891, 15)
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  


titanic 주요 열: survived(생존 0/1) · pclass(객실 등급 1~3) · sex(성별) · age(나이, 결측 있음) · fare(요금) · embark_town(탑승 항구) · alone(혼자 탑승 True/False)

### 1. 열 선택

In [220]:
# fare 열 하나를 선택해 타입을 출력하세요.

type(titanic["fare"])

pandas.Series

In [221]:
# survived, pclass, sex, age 4개 열을 선택해 타입과 shape를 출력하세요.

ti_col = titanic[["survived","pclass","sex","age"]]
print(type(ti_col))
print(ti_col.shape)

<class 'pandas.DataFrame'>
(891, 4)


In [222]:
# 1번과 2번의 타입이 다른 이유를 주석 한 줄로 쓰세요.
# => 1번은 하나의 컬럼으로 시리즈이고, 2번은 여러개의 컬럼의 시리즈 조합으로 데이터프레임 이라서

### 2. loc vs iloc

In [223]:
# loc으로 인덱스 10~15 행의 sex, age 열을 선택해 출력하세요.
ti_loc = titanic.loc[10:15,["sex","age"]]
ti_loc

,sex,age
10,female,4.0
11,female,58.0
12,male,20.0
13,male,39.0
14,female,14.0
15,female,55.0


In [224]:
# iloc으로 같은 열(sex는 2번째, age는 3번째 위치)을 행 위치 10:15로 선택해 출력하세요.
ti_iloc = titanic.iloc[10:15,2:4]
ti_iloc

,sex,age
10,female,4.0
11,female,58.0
12,male,20.0
13,male,39.0
14,female,14.0


In [225]:
# 두 결과의 행 수를 각각 출력하고, 왜 다른지 주석으로 쓰세요.
print(len(ti_loc))      # loc는 행과 열의 슬라이싱 2번째 범위를 포함한다
print(len(ti_iloc))     # iloc는 행과 열의 슬라이싱 2번째 범위를 포함하지 않는다


6
5


### 3. iloc 응용

In [226]:
# iloc으로 마지막 3행의 앞 4개 열만 선택해 출력하세요. (힌트: 음수 위치)

titanic.iloc[:-4:-1, 0:4]

,survived,pclass,sex,age
890,0,3,male,32.0
889,1,1,male,26.0
888,0,3,female,NaN


### 4. 조건 마스크

In [227]:
# 나이가 10살 미만인지 검사하는 마스크를 만들어 head(5)를 출력하세요.
mask = titanic["age"] < 10
print("1. ",titanic[mask].head(5))


1.      survived  pclass     sex  age  sibsp  parch     fare embarked   class  \
7          0       3    male  2.0      3      1  21.0750        S   Third   
10         1       3  female  4.0      1      1  16.7000        S   Third   
16         0       3    male  2.0      4      1  29.1250        Q   Third   
24         0       3  female  8.0      3      1  21.0750        S   Third   
43         1       2  female  3.0      1      2  41.5792        C  Second   

      who  adult_male deck  embark_town alive  alone  
7   child       False  NaN  Southampton    no  False  
10  child       False    G  Southampton   yes  False  
16  child       False  NaN   Queenstown    no  False  
24  child       False  NaN  Southampton    no  False  
43  child       False  NaN    Cherbourg   yes  False  


In [228]:
# 마스크로 10살 미만 승객 수를 출력하세요.
mask = titanic["age"] < 10
print("2. count:",len(titanic[mask]))
# titanic[mask].count

2. count: 62


### 5. 복합 조건

In [229]:
# 각각 조건을 만족하는 행 수를 출력하세요.

# 1. 3등석 **그리고** 남성 **그리고** 생존한 승객 (`&`)
mask = (titanic["pclass"] == 3) & (titanic["sex"] == "male") & (titanic["survived"] == 1)
# print(titanic[mask][["survived","pclass","sex"]])
print("1. count: ",len(titanic[mask][["survived","pclass","sex"]]))

# 2. 1등석 **또는** 요금 100 이상인 승객 (`|`)
mask = (titanic["pclass"] == 1) | (titanic["fare"] >= 100)
print("2. count: ",len(titanic[mask]))

# 3. 혼자 탑승하지 **않은** 승객 (`~`, `alone` 열 사용)
mask = ~(titanic["alone"] == True)
print("3. count: ",len(titanic[mask]))

1. count:  47
2. count:  216
3. count:  354


### 6. isin · between

In [230]:
# isin으로 Cherbourg 또는 Queenstown에서 탑승한 승객을 골라 shape를 출력하세요.
mask = titanic["embark_town"].isin(["Cherbourg","Queenstown"])
print("1. shape:", titanic[mask].shape)


# between으로 나이 20~30세(양 끝 포함) 승객을 골라 shape와 나이의 min·max를 출력하세요.
mask = titanic["age"].between(20,30)
x = titanic[mask]
print("2. shape:",x.shape)
print("min: ",x["age"].min())
print("min: ",x["age"].max())

# 같은 범위를 양 끝 미포함(inclusive="neither")으로 바꿔 shape를 출력하세요.
mask = titanic["age"].between(20,30,inclusive="neither")
x = titanic[mask]
print("3. shape:",x.shape)
print("min: ",x["age"].min())
print("min: ",x["age"].max())


1. shape: (245, 15)
2. shape: (245, 15)
min:  20.0
min:  30.0
3. shape: (205, 15)
min:  20.5
min:  29.0


### 7. loc[조건, 열]

In [231]:
# 12살 이하 그리고 3등석 승객의 age, sex, survived 열만 kids 변수에 담고
# 행 수와 생존률(survived 평균, 소수 둘째 자리)을 출력하세요.

mask = (titanic["age"] <= 12) & (titanic["pclass"] == 3)
kids = titanic[mask].loc[:,["age","sex","survived"]]
print("count:",len(kids))
print("mean:",round(kids["survived"].mean(),2))

count: 48
mean: 0.42


### 8. query

In [232]:
# 변수 min_fare = 50을 만들고,
# query로 Cherbourg 탑승 그리고 요금이 min_fare 이상인 승객의 shape를 출력하세요.
min_fare = 50
print("shape:",titanic.query("embark_town == 'Cherbourg' and fare >= @min_fare").shape)


# 5-1번 조건을 query로 다시 써서 행 수가 같은지 확인하세요.
# 3등석 그리고 남성 그리고 생존한 승객
print("count:",len(titanic.query("pclass == 3 and survived == 1 and sex == 'male'")))


shape: (61, 15)
count: 47


### 9. 정렬

In [233]:
# pclass 오름차순, 같은 등급 안에서는 fare 내림차순으로 정렬해 pclass, fare, sex, age 열의 상위 3행을 출력하세요.

print(titanic[["pclass","fare","sex","age"]]
      .sort_values(by= ["pclass","fare"], ascending=[True,False])
      .head(3))

     pclass      fare     sex   age
258       1  512.3292  female  35.0
679       1  512.3292    male  36.0
737       1  512.3292    male  35.0


In [234]:
# 요금 내림차순으로 정렬한 결과를 변수에 담고,
# iloc[0]과 loc[0]의 fare 값을 각각 출력하세요.
# 두 값이 다른 이유를 주석으로 쓰세요.

mask = titanic.sort_values(by= "fare", ascending=False)
print("fare: ",mask["fare"].iloc[0]) # 정렬 후 0번째 인덱스를 찾아감
print("fare: ",mask["fare"].loc[0]) # name 으로 찾아가기때문에 정렬 하더라도 titanic의 0번째 의 fare 값을 출력함



fare:  512.3292
fare:  7.25


### 10. 종합 — 20~40세 남성, 1·2등석 vs 3등석

In [ ]:
# 질문: "20~40세 남성 승객의 생존률은 객실 등급에 따라 얼마나 다를까?"
# 1. 남성 · 나이 20~40세(between) · 1·2등석(isin) 조건으로 pclass, age, fare, survived 열을 선택하세요.
mask = (titanic["age"].between(20,40)) & (titanic["pclass"].isin([1,2]))
print(titanic.loc[mask,["pclass","age","fare","survived"]])

# 2. 행 수, 생존률(소수 둘째 자리), 요금 상위 3명을 출력하세요.
print("row count:",
      len(titanic[mask]))

print("survived mean(1,2 pclass):",
      round(titanic.loc[mask,"survived"].mean()*100,2),"%")

print(titanic[mask].sort_values(by="fare",ascending=False).head(3))

# 3. 같은 조건을 query로 써서 행 수를 검증하세요.
print("row count:",
      len(titanic.query("age >= 20 and age <= 40 and pclass == [1,2]")))

# 4. 등급만 3등석으로 바꿔 생존률을 구하고, 두 생존률을 비교하는 해석을 마크다운 셀에 2~3문장으로 쓰세요.
print("survived mean(3 pclass):",
      round(titanic.query("age >= 20 and age <= 40 and pclass == 3")["survived"].mean()*100,2),"%")



     pclass   age     fare  survived
1         1  38.0  71.2833         1
3         1  35.0  53.1000         1
20        2  35.0  26.0000         0
21        2  34.0  13.0000         1
23        1  28.0  35.5000         1
..      ...   ...      ...       ...
874       2  28.0  24.0000         1
880       2  25.0  26.0000         1
883       2  28.0  10.5000         0
886       2  27.0  13.0000         0
889       1  26.0  30.0000         1

[193 rows x 4 columns]
row count: 193
survived mean(1,2 pclass): 56.99 %
     survived  pclass     sex   age  sibsp  parch      fare embarked  class  \
258         1       1  female  35.0      0      0  512.3292        C  First   
679         1       1    male  36.0      0      1  512.3292        C  First   
737         1       1    male  35.0      0      0  512.3292        C  First   

       who  adult_male deck embark_town alive  alone  
258  woman       False  NaN   Cherbourg   yes   True  
679    man        True    B   Cherbourg   yes  False  


객실등급 1,2등급 20~40세 남성은 생존율이 56.99% 고,

객실등급 3등급 20~40세 남성은 생존율이 22.99% 로

객실 등급이 높을수록 생존율이 높았다고 할 수 있다.